# TP 1 (Transformer) - Du texte brut au modele bigramme

**Cours NLP et Transformers - CY Tech**

Objectif : comprendre comment un corpus devient un ensemble de predictions
supervisees, puis entrainer une table qui predit le caractere suivant.

Etapes :
1. telecharger et inspecter le corpus ;
2. passer des caracteres aux entiers (encode / decode) ;
3. former contextes, cibles et mini-batchs ;
4. programmer le modele bigramme (logits) ;
5. generer, entrainer, puis comparer.

In [1]:
import os
import urllib.request

import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

## Etape 1 - Telecharger et inspecter le corpus

Le modele n'apprend pas Shakespeare : il recoit une longue suite de caracteres.
L'unite d'un token est ici un caractere. `encoding="utf-8"` garantit une lecture
correcte du fichier.

In [2]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

with open("input.txt", "r", encoding="utf-8") as fichier:
    texte = fichier.read()

print(len(texte), "caracteres")
print(texte[:200])

1115394 caracteres
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## Etape 2 - Passer des caracteres aux entiers

On construit un vocabulaire trie (pour que l'encodage soit reproductible), puis
deux dictionnaires parfaitement inverses. Un identifiant n'a pas de sens par
lui-meme : il designe seulement une ligne dans une table.

In [3]:
caracteres = sorted(list(set(texte)))
taille_vocabulaire = len(caracteres)
print("taille du vocabulaire :", taille_vocabulaire)

stoi = {caractere: i for i, caractere in enumerate(caracteres)}
itos = {i: caractere for i, caractere in enumerate(caracteres)}

encoder = lambda chaine: [stoi[c] for c in chaine]
decoder = lambda identifiants: "".join([itos[i] for i in identifiants])

message = "hii there"
print(encoder(message))
print(decoder(encoder(message)))

taille du vocabulaire : 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


## Etape 3 - Fabriquer les predictions attendues

Une sequence de longueur T+1 fournit une entree (x) et une cible (y) decalee
d'une position : chaque token doit predire le suivant. On separe aussi train et
validation pour mesurer la generalisation.

In [4]:
donnees = torch.tensor(encoder(texte), dtype=torch.long)
n = int(0.9 * len(donnees))
donnees_train = donnees[:n]
donnees_val = donnees[n:]

block_size = 8
x = donnees_train[:block_size]
y = donnees_train[1:block_size + 1]
for t in range(block_size):
    contexte = x[:t + 1]
    cible = y[t]
    print(contexte.tolist(), "->", cible.item())

[18] -> 47
[18, 47] -> 56
[18, 47, 56] -> 57
[18, 47, 56, 57] -> 58
[18, 47, 56, 57, 58] -> 1
[18, 47, 56, 57, 58, 1] -> 15
[18, 47, 56, 57, 58, 1, 15] -> 47
[18, 47, 56, 57, 58, 1, 15, 47] -> 58


## Etape 4 - Echantillonner un mini-batch

Un mini-batch regroupe B sequences independantes de longueur T. Toutes les
predictions seront calculees en parallele. On retire `block_size` a la borne de
tirage pour toujours disposer d'une cible decalee.

In [5]:
batch_size = 4
block_size = 8

def get_batch(partie):
    source = donnees_train if partie == "train" else donnees_val
    indices = torch.randint(len(source) - block_size, (batch_size,))
    x = torch.stack([source[i:i + block_size] for i in indices])
    y = torch.stack([source[i + 1:i + block_size + 1] for i in indices])
    return x, y

xb, yb = get_batch("train")
print("x :", xb.shape)
print("y :", yb.shape)

x : torch.Size([4, 8])
y : torch.Size([4, 8])


## Etape 5 - Une table produit les logits

Chaque caractere choisit directement une ligne contenant un score pour chacun
des C caracteres possibles. On passe donc des indices (B, T) aux logits
(B, T, C). La perte est l'entropie croisee entre les logits et les cibles.

In [6]:
class ModeleBigramme(nn.Module):
    def __init__(self, taille_vocabulaire):
        super().__init__()
        self.table_embedding = nn.Embedding(taille_vocabulaire, taille_vocabulaire)

    def forward(self, indices, cibles=None):
        logits = self.table_embedding(indices)
        if cibles is None:
            perte = None
        else:
            B, T, C = logits.shape
            logits_plats = logits.view(B * T, C)
            cibles_plates = cibles.view(B * T)
            perte = F.cross_entropy(logits_plats, cibles_plates)
        return logits, perte

    def generer(self, indices, nombre_tokens):
        for _ in range(nombre_tokens):
            logits, _ = self(indices)
            logits = logits[:, -1, :]
            probabilites = F.softmax(logits, dim=-1)
            indice_suivant = torch.multinomial(probabilites, num_samples=1)
            indices = torch.cat((indices, indice_suivant), dim=1)
        return indices

modele = ModeleBigramme(taille_vocabulaire)
logits, perte = modele(xb, yb)
print("logits :", logits.shape)
print("perte initiale :", round(perte.item(), 3))

logits : torch.Size([4, 8, 65])
perte initiale : 5.036


## Etapes 6 et 7 - Generer, puis apprendre

On genere d'abord avec les poids aleatoires (texte incoherent), puis on entraine
la table et on relance exactement la meme generation. Pendant la generation on
ne garde que le dernier pas de temps car le bigramme ne regarde que le token
courant.

In [7]:
contexte = torch.zeros((1, 1), dtype=torch.long)
print("--- avant entrainement ---")
print(decoder(modele.generer(contexte, 200)[0].tolist()))

--- avant entrainement ---

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheHd:.?nA!LgjgL&jcM-sZW!HFsWt'rv.x;z&hPyd'XHOxnAcj.erRjcLxxVFSrOpW.oeGJ Ai$yi&!mRfeopx,f.e..iBgxVrl-VTh


In [8]:
optimiseur = torch.optim.AdamW(modele.parameters(), lr=1e-3)

for pas in range(3000):
    xb, yb = get_batch("train")
    logits, perte = modele(xb, yb)
    optimiseur.zero_grad(set_to_none=True)
    perte.backward()
    optimiseur.step()

print("perte finale :", round(perte.item(), 3))

perte finale : 3.204


In [9]:
print("--- apres entrainement ---")
print(decoder(modele.generer(contexte, 200)[0].tolist()))

--- apres entrainement ---

Teregx:HHfheXDXqwF: myEd CarPEM
FWjxu t?besternA'm,
L
UVrrotopkig
to bot knCXVRVDKZEDJl-ziuatw:LLII'fd.S:LROV?GLwaz.Hf:UCA,
T;HA$  WtvBe BO,
kCaEYcoobAr vDU..ctkhqIUng,OfentFF py;ARNh d llEQQ:uTuc?ers


## Bilan

- Les tokens sont des entiers car un reseau manipule des nombres, pas des
  caracteres.
- x et y sont decales d'une position : chaque position predit la suivante.
- B, T, C designent le nombre de sequences, la longueur de contexte et la taille
  du vocabulaire.
- Les logits deviennent des probabilites via softmax.
- L'entrainement ajuste la table pour rendre les suites reelles plus probables :
  la perte baisse et le texte devient plus credible, mais le bigramme ne regarde
  qu'un seul caractere en arriere, ce qui limite la coherence.